In [1]:
!pip install pandas requests

In [2]:
import pandas as pd
import requests
from io import StringIO

SEASONS = ["1516", "1617", "1718", "1819", "1920", "2021", "2122", "2223", "2324", "2425"]
BASE_URL = "https://www.football-data.co.uk/mmz4281/{season}/T1.csv"

def load_match_data():
    all_dfs = []

    for season in SEASONS:
        url = BASE_URL.format(season=season)
        print(f"Downloading season {season}...")

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        df = pd.read_csv(StringIO(response.text))
        df["season_code"] = season
        all_dfs.append(df)

    matches = pd.concat(all_dfs, ignore_index=True)
    return matches

def prepare_besiktas_home_matches(matches):
    home = matches[matches["HomeTeam"].astype(str).str.strip() == "Besiktas"].copy()

    cols = [
        "Date", "Time", "HomeTeam", "AwayTeam",
        "FTHG", "FTAG", "FTR",
        "HS", "AS", "HST", "AST",
        "HF", "AF", "HY", "AY", "HR", "AR",
        "season_code"
    ]

    existing_cols = [col for col in cols if col in home.columns]
    home = home[existing_cols].copy()

    home["Date"] = pd.to_datetime(home["Date"], dayfirst=True, errors="coerce")
    home["Time"] = home["Time"].fillna("19:00")

    home["kickoff_local"] = pd.to_datetime(
        home["Date"].dt.strftime("%Y-%m-%d") + " " + home["Time"],
        errors="coerce"
    )

    home["besiktas_goals"] = home["FTHG"]
    home["opponent_goals"] = home["FTAG"]
    home["total_goals"] = home["FTHG"] + home["FTAG"]

    home = home.sort_values("kickoff_local").reset_index(drop=True)
    return home

def load_weather_data(start_date, end_date):
    params = {
        "latitude": 41.0422,
        "longitude": 29.0083,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,wind_speed_10m",
        "timezone": "Europe/Istanbul"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=60
    )
    response.raise_for_status()

    weather_json = response.json()
    weather = pd.DataFrame(weather_json["hourly"])
    weather["time"] = pd.to_datetime(weather["time"])
    return weather

def merge_data(home, weather):
    home = home.copy()
    home["kickoff_hour"] = home["kickoff_local"].dt.floor("h")

    merged = home.merge(
        weather,
        left_on="kickoff_hour",
        right_on="time",
        how="left"
    )

    merged["rainy_match"] = (merged["precipitation"].fillna(0) > 0).astype(int)
    merged["heavy_rain_match"] = (merged["precipitation"].fillna(0) >= 2).astype(int)

    merged = merged.dropna(subset=["Date", "AwayTeam", "FTHG", "FTAG", "total_goals"])
    merged = merged.reset_index(drop=True)

    return merged

matches = load_match_data()
home = prepare_besiktas_home_matches(matches)

print("Match count:", len(home))
print("Date range:", home["Date"].min(), "->", home["Date"].max())

start_date = home["Date"].min().strftime("%Y-%m-%d")
end_date = home["Date"].max().strftime("%Y-%m-%d")

weather = load_weather_data(start_date, end_date)
merged = merge_data(home, weather)

merged.to_csv("clean_data.csv", index=False)

print("Done.")
print("Final shape:", merged.shape)
print(merged.head())

/tmp/ipykernel_3446/2528414697.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  home["Date"] = pd.to_datetime(home["Date"], dayfirst=True, errors="coerce")


Match count: 179
Date range: 2015-08-22 00:00:00 -> 2025-05-25 00:00:00
Done.
Final shape: (179, 30)
        Date   Time  HomeTeam     AwayTeam  FTHG  FTAG FTR  HS  AS  HST  ...  \
0 2015-08-22  19:00  Besiktas  Trabzonspor   1.0   2.0   A NaN NaN  NaN  ...   
1 2015-09-13  19:00  Besiktas   Buyuksehyr   2.0   0.0   H NaN NaN  NaN  ...   
2 2015-09-27  19:00  Besiktas   Fenerbahce   3.0   2.0   H NaN NaN  NaN  ...   
3 2015-10-18  19:00  Besiktas     Rizespor   1.0   0.0   H NaN NaN  NaN  ...   
4 2015-10-30  19:00  Besiktas    Kasimpasa   3.0   3.0   D NaN NaN  NaN  ...   

   opponent_goals  total_goals        kickoff_hour                time  \
0             2.0          3.0 2015-08-22 19:00:00 2015-08-22 19:00:00   
1             0.0          2.0 2015-09-13 19:00:00 2015-09-13 19:00:00   
2             2.0          5.0 2015-09-27 19:00:00 2015-09-27 19:00:00   
3             0.0          1.0 2015-10-18 19:00:00 2015-10-18 19:00:00   
4             3.0          6.0 2015-10-30 19:00: